In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Suapan balik klasik dan aliran kawalan (litar dinamik)

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



# Suapan balik klasik dan aliran kawalan


<details>
<summary><b>Versi pakej</b></summary>

Kod pada halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan penggunaan versi ini atau yang lebih baharu.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
> **Note:** Versi baharu litar dinamik kini tersedia kepada semua pengguna pada semua Backend. Anda boleh menjalankan litar dinamik pada skala utiliti. Lihat [pengumuman ini](/announcements/product-updates/2025-09-25-new-dynamic-circuits) untuk maklumat lanjut.

Litar dinamik ialah alat yang berkuasa yang membolehkan anda mengukur Qubit di tengah-tengah pelaksanaan Circuit kuantum dan kemudian melakukan operasi logik klasik dalam Circuit, berdasarkan keputusan pengukuran mid-Circuit tersebut. Proses ini juga dikenali sebagai _suapan balik klasik_. Walaupun ini masih peringkat awal dalam memahami cara terbaik memanfaatkan litar dinamik, komuniti penyelidikan kuantum telah mengenal pasti beberapa kes penggunaan, seperti berikut:

* Penyediaan keadaan kuantum yang cekap, seperti [keadaan GHZ,](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) [keadaan-W,](https://arxiv.org/abs/2403.07604) (untuk maklumat lanjut tentang keadaan-W, rujuk juga ["State preparation by shallow circuits using feed forward"](https://arxiv.org/abs/2307.14840)) dan kelas luas [matrix product states](https://arxiv.org/abs/2404.16083)
* [Keterikatan jarak jauh yang cekap](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) antara Qubit pada cip yang sama dengan menggunakan Circuit cetek
* [Pensampelan cekap litar seperti IQP](https://arxiv.org/pdf/2505.04705)

Namun, penambahbaikan yang dibawa oleh litar dinamik ini disertai dengan pertukaran. Pengukuran mid-Circuit dan operasi klasik biasanya mempunyai masa pelaksanaan yang lebih lama berbanding Gate dua-Qubit, dan peningkatan masa ini mungkin menghapuskan manfaat daripada kedalaman Circuit yang berkurangan. Oleh itu, mengurangkan panjang pengukuran mid-Circuit menjadi fokus penambahbaikan semasa IBM Quantum&reg; mengeluarkan [versi baharu](/announcements/product-updates/2025-03-03-new-version-dynamic-circuits) litar dinamik.

[Spesifikasi OpenQASM 3](https://openqasm.com/language/classical.html#looping-and-branching) mentakrifkan beberapa struktur aliran kawalan, tetapi Qiskit Runtime pada masa ini hanya menyokong pernyataan `if` bersyarat. Dalam Qiskit SDK, ini berpadanan dengan kaedah [if_test](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#if_test) pada [QuantumCircuit.](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) Kaedah ini mengembalikan [pengurus konteks](https://docs.python.org/3/reference/datamodel.html#with-statement-context-managers) dan biasanya digunakan dalam pernyataan `with`. Panduan ini menerangkan cara menggunakan pernyataan bersyarat ini.

> **Note:** Contoh kod dalam panduan ini menggunakan arahan ukur standard untuk pengukuran mid-Circuit. Walau bagaimanapun, adalah disyorkan agar anda menggunakan arahan [`MidCircuitMeasure`](/guides/measure-qubits#midcircuit) sebaliknya, jika Backend menyokongnya. Lihat [dokumentasi Pengukuran mid-Circuit](/guides/measure-qubits#mid-circuit-measurements) untuk butiran.
## Pernyataan `if`
Pernyataan `if` digunakan untuk melaksanakan operasi secara bersyarat berdasarkan nilai bit atau daftar klasik.

Dalam contoh di bawah, kami menggunakan Gate Hadamard pada sebuah Qubit dan mengukurnya. Jika hasilnya ialah 1, maka kami menggunakan Gate X pada Qubit tersebut, yang berkesan membalikkannya semula ke keadaan 0. Kemudian kami mengukur Qubit itu sekali lagi. Hasil pengukuran seharusnya ialah 0 dengan kebarangkalian 100%.

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
# Use MidCircuitMeasure() if it's supported by the backend.
# circuit.append(MidCircuitMeasure(), [q0], [c0])
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)):
    circuit.x(q0)
circuit.measure(q0, c0)
circuit.draw("mpl")

# example output counts: {'0': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg)

Pernyataan `with` boleh diberi sasaran tugasan yang itu sendiri merupakan pengurus konteks yang boleh disimpan dan kemudiannya digunakan untuk mencipta blok else, yang dilaksanakan apabila kandungan blok `if` *tidak* dilaksanakan.

Dalam contoh di bawah, kami memulakan daftar dengan dua Qubit dan dua bit klasik. Kami menggunakan Gate Hadamard pada Qubit pertama dan mengukurnya. Jika hasilnya ialah 1, maka kami menggunakan Gate Hadamard pada Qubit kedua; jika tidak, kami menggunakan Gate X pada Qubit kedua. Akhirnya, kami mengukur Qubit kedua juga.

In [2]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1) = qubits
(c0, c1) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)) as else_:
    circuit.h(q1)
with else_:
    circuit.x(q1)
circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 260, '11': 272, '10': 492}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg)

Selain daripada bersyarat pada satu bit klasik, adalah juga boleh bersyarat pada nilai daftar klasik yang terdiri daripada beberapa bit.

Dalam contoh di bawah, kami menggunakan Gate Hadamard pada dua Qubit dan mengukurnya. Jika hasilnya ialah `01`, iaitu Qubit pertama ialah 1 dan Qubit kedua ialah 0, maka kami menggunakan Gate X pada Qubit ketiga. Akhirnya, kami mengukur Qubit ketiga. Perhatikan bahawa untuk kejelasan, kami memilih untuk menentukan keadaan bit klasik ketiga, iaitu 0, dalam syarat `if`. Dalam lukisan Circuit, syarat ditunjukkan oleh bulatan pada bit klasik yang dijadikan syarat. Bulatan hitam menunjukkan syarat pada 1, manakala bulatan putih menunjukkan syarat pada 0.

In [3]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.if_test((clbits, 0b001)):
    circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 269, '011': 260, '000': 252, '010': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg)

## Ungkapan klasik
Modul ungkapan klasik Qiskit [`qiskit.circuit.classical`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical) mengandungi representasi eksperimentasi operasi masa larian pada nilai klasik semasa pelaksanaan Circuit. Disebabkan had perkakasan, hanya syarat `QuantumCircuit.if_test()` yang disokong pada masa ini.

Contoh berikut menunjukkan bahawa anda boleh menggunakan pengiraan pariti untuk mencipta keadaan GHZ n-Qubit menggunakan litar dinamik. Pertama, jana $n/2$ pasangan Bell pada Qubit bersebelahan. Kemudian, sambungkan pasangan-pasangan ini menggunakan lapisan Gate CNOT di antara pasangan. Anda kemudian mengukur Qubit sasaran semua Gate CNOT sebelumnya dan menetapkan semula setiap Qubit yang diukur ke keadaan $\vert 0 \rangle$. Anda menggunakan $X$ pada setiap tapak yang tidak diukur di mana pariti semua bit sebelumnya adalah ganjil. Akhirnya, Gate CNOT digunakan pada Qubit yang diukur untuk memulihkan keterikatan yang hilang dalam pengukuran.

Dalam pengiraan pariti, elemen pertama ungkapan yang dibina melibatkan pengangkatan objek Python `mr[0]` ke nod [`Value`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical#value) (`lift` digunakan untuk menukar objek arbitrari kepada ungkapan klasik). Ini tidak diperlukan untuk `mr[1]` dan daftar klasik yang mungkin berikutnya, kerana ia adalah input kepada `expr.bit_xor`, dan sebarang pengangkatan yang diperlukan dilakukan secara automatik dalam kes-kes ini. Ungkapan sedemikian boleh dibina dalam gelung dan konstruk lain.

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

num_qubits = 8
if num_qubits % 2 or num_qubits < 4:
    raise ValueError("num_qubits must be an even integer ≥ 4")
meas_qubits = list(range(2, num_qubits, 2))  # qubits to measure and reset

qr = QuantumRegister(num_qubits, "qr")
mr = ClassicalRegister(len(meas_qubits), "m")
qc = QuantumCircuit(qr, mr)

# Create local Bell pairs
qc.reset(qr)
qc.h(qr[::2])
for ctrl in range(0, num_qubits, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Glue neighboring pairs
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Measure boundary qubits between pairs,reset to 0
for k, q in enumerate(meas_qubits):
    qc.measure(qr[q], mr[k])
    qc.reset(qr[q])

# Parity-conditioned X corrections
# Each non-measured qubit gets flipped iff the parity (XOR) of all
# preceding measurement bits is 1
for tgt in range(num_qubits):
    if tgt in meas_qubits:  # skip measured qubits
        continue
    # all measurement registers whose physical qubit index < tgt
    left_bits = [k for k, q in enumerate(meas_qubits) if q < tgt]
    if not left_bits:  # skip if list empty
        continue

    # build XOR-parity expression
    parity = expr.lift(
        mr[left_bits[0]]
    )  # lift the first bit to Value so it will be treated like a boolean.
    for k in left_bits[1:]:
        parity = expr.bit_xor(
            mr[k], parity
        )  # calculate parity with all other bits
    with qc.if_test(parity):  # Add X if parity is 1
        qc.x(qr[tgt])

# Re-entangle measured qubits
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

In [5]:
qc.draw(output="mpl", style="iqp", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg)

## Cari Backend yang menyokong litar dinamik
Untuk mencari semua Backend yang boleh diakses oleh akaun anda dan menyokong litar dinamik, jalankan kod seperti berikut. Contoh ini mengandaikan bahawa anda telah [menyimpan kelayakan log masuk anda.](/guides/save-credentials) Anda juga boleh [menentukan kelayakan secara eksplisit](/guides/initialize-account#explicit) semasa memulakan akaun perkhidmatan Qiskit Runtime anda. Ini membolehkan anda melihat Backend yang tersedia pada instans atau jenis pelan tertentu, contohnya.

> **Note:** - Backend yang tersedia untuk akaun bergantung pada instans yang ditentukan dalam kelayakan.
> - Versi baharu litar dinamik kini tersedia kepada semua pengguna pada semua Backend. Lihat [pengumuman ini](/announcements/product-updates/2025-09-25-new-dynamic-circuits) untuk maklumat lanjut.

In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
dc_backends = service.backends(dynamic_circuits=True)
print(dc_backends)

[<IBMBackend('ibm_pittsburgh')>, <IBMBackend('ibm_boston')>, <IBMBackend('ibm_fez')>, <IBMBackend('ibm_miami')>, <IBMBackend('ibm_marrakesh')>, <IBMBackend('ibm_torino')>, <IBMBackend('ibm_kingston')>]


## Qiskit Runtime limitations

Be aware of the following constraints when running dynamic circuits in Qiskit Runtime.

- Due to the limited physical memory on control electronics, there is also a limit on the number of `if` statements and size of their operands. This limit is a function of the number of broadcasts and the number of broadcasted bits in a job (not a circuit).

   When processing an `if` condition, measurement data needs to be transferred to the control logic to make that evaluation. A broadcast is a transfer of unique classical data, and broadcasted bits is the number of classical bits being transferred. Consider the following:

   ```python
   c0 = ClassicalRegister(3)
   c1 = ClassicalRegister(5)
   ...
   with circuit.if_test((c0, 1)) ...
   with circuit.if_test((c0, 3)) ...
   with circuit.if_test((c1[2], 1)) ...
   ```
   In the previous code example, the first two `if_test` objects on `c0` are considered one broadcast because the content of `c0` did not change, and thus does not need to be re-broadcasted. The `if_test` on `c1` is a second broadcast. The first one broadcasts all three bits in `c0` and the second broadcasts just one bit, making a total of four broadcasted bits.

   Currently, if you broadcast 60 bits each time, then the job can have approximately 300 broadcasts. If you broadcast just one bit each time, however, then the job can have 2400 broadcasts.

- The operand used in an `if_test` statement must be 32 or fewer bits. Thus, if you are comparing an entire `ClassicalRegister`, the size of that `ClassicalRegister` must be 32 or fewer bits. If you are comparing just a single bit from a `ClassicalRegister`, however, that `ClassicalRegister` can be of any size (since the operand is only one bit).

   For example, the "Not valid" code block does not work because `cr` is more than 32 bits.  You can, however, use a classical register wider than 32 bits if you are testing only one bit, as shown in the "Valid" code block.

   <Tabs>
   <TabItem value="Not valid" label="Not valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr, 15)):
            ...
      ```
   </TabItem>
   <TabItem value="Valid" label="Valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr[5], 1)):
            ...
      ```
   </TabItem>
   </Tabs>

- Nested conditionals are not allowed. For example, the following code block will not work because it has an `if_test` inside another `if_test`:
   <Tabs>
    <TabItem value="Not valid" label="Not valid">
        ```python
           c1 = ClassicalRegister(1, "c1")
           c2 = ClassicalRegister(2, "c2")
           ...
           with circ.if_test((c1, 1)):
            with circ.if_test(c2, 1)):
             ...
        ```
     </TabItem>
     <TabItem value="Valid" label="Valid">
        ```python
        cr = ClassicalRegister(2)
        ...
        with circuit.if_test((cr, 0b11)):
          ...
        ```
     </TabItem>
    </Tabs>

- Having `reset` or measurements inside conditionals is not supported.
- Arithmetic operations are not supported.
- See the [OpenQASM 3 feature table](/docs/guides/qasm-feature-table) to determine which OpenQASM 3 features are supported on Qiskit and Qiskit Runtime.
- When OpenQASM 3 (instead of `QuantumCircuit`) is used as the input format to pass circuits to Qiskit Runtime primitives, only instructions that can be loaded into Qiskit are supported. Classical operations, for example, are not supported because they cannot be loaded into Qiskit. See [Import an OpenQASM 3 program into Qiskit](/docs/guides/interoperate-qiskit-qasm3#import-an-openqasm-3-program-into-qiskit) for more information.
- The `for`, `while`, and `switch` instructions are not supported.

## Use dynamic circuits with Estimator

Since Estimator does not support dynamic circuits, you can use Sampler and build your own measurement circuits instead. Alternatively, you can use the [Executor primitive,](/docs/guides/directed-execution-model#executor-primitive) which supports dynamic circuits.

To replicate Estimator's behavior, follow this process:

1. Group the terms of all observables into a partition.  This can be done by using the [`PauliList` API,](/docs/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting) for example.
     <Admonition type="note">
      You can use the [`BitArray`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.primitives.BitArray#expectation_values) primitive attribute to compute the expectation values of the provided observables.
     </Admonition>
2. Execute one basis change circuit per partition (whichever basis change needs to be done for each partition). See the Measurement bases addon utility  [`measurement_bases` module](https://github.com/Qiskit/qiskit-addon-utils/blob/38ea05431f2e9830adf4ec9265f6d19758a32096/qiskit_addon_utils/exp_vals/measurement_bases.py) for more information. [Get started with utilities.](/docs/guides/qiskit-addons-utils#get-started-with-utilities)
3. Add back together the results for each partition.

## Next steps

<Admonition type="tip" title="Recommendations">
- Learn how to implement accurate dynamic decoupling by using [stretch.](/docs/guides/stretch)
- Learn about the shorter [mid-circuit measurements](/docs/guides/measure-qubits#mid-circuit-measurements) that reduce the circuit time.
- Use [circuit schedule visualization](/docs/guides/visualize-circuit-timing#qiskit-runtime-support) to debug and optimize your dynamic circuits.
</Admonition>